# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50

# Cargamos el dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalización de los píxeles al rango [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Conversión de etiquetas a vectores categóricos (One-Hot Encoding)
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

# Configuración de Data Augmentation para robustecer el entrenamiento
datagen = ImageDataGenerator(
    rotation_range=15,       # Rotación aleatoria de imágenes
    width_shift_range=0.1,   # Desplazamiento horizontal
    height_shift_range=0.1,  # Desplazamiento vertical
    horizontal_flip=True,    # Volteo horizontal aleatorio
    fill_mode='nearest'
)
datagen.fit(x_train)



## Definiciones de modelos

In [6]:
# Recuerda aquí solo generar las arquitecturas, cada capa así como sus neuronas.
def build_cnn_light():
    """Arquitectura 1: CNN Ligera con regularización estándar"""
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        
        # Bloque Convolucional 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        # Bloque Convolucional 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Clasificador Denso
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ], name="CNN_Ligera")
    return model


def build_cnn_deep():
    """Arquitectura 2: CNN Profunda con bloques acumulados (Estilo VGG)"""
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        
        # Bloque Convolucional 1 (Doble capa)
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Bloque Convolucional 2 (Aumento de filtros)
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.35),
        
        # Bloque Convolucional 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),
        
        # Clasificador Denso Avanzado
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name="CNN_Profunda")
    return model


def build_transfer_learning():
    """Arquitectura 3: Transfer Learning utilizando VGG16 (Pesos de ImageNet)"""
    # Cargamos VGG16 sin el clasificador superior (top) adaptado a 32x32x3
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
    
    # Congelamos las capas convolucionales base para preservar el preentrenamiento
    base_model.trainable = False
    
    model = keras.Sequential([
        base_model,
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ], name="Transfer_Learning_VGG16")
    return model

# Instanciamos los modelos creados
model_light = build_cnn_light()
model_deep = build_cnn_deep()
model_tl = build_transfer_learning()

58889256/58889256 [==============================] - 11s 0us/step


## Entrenamiento de modelos.

In [ ]:
# Aquí agrega la compilación y entrenamiento de las arquitecturas generadas.
# Configuración base de entrenamiento
batch_size = 64
epochs = 25  # Puedes aumentarlo a 40-50 si tienes GPU activa para mejor convergencia
histories = {}

models_dict = {
    "CNN_Ligera": model_light,
    "CNN_Profunda": model_deep,
    "Transfer_Learning": model_tl
}

# Callback de Early Stopping para detener el entrenamiento si deja de mejorar
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

for name, model in models_dict.items():
    print(f"\n==================== Compilando y Entrenando: {name} ====================")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Entrenamiento aplicando Data Augmentation de forma dinámica en cada batch
    history = model.fit(
        datagen.flow(x_train, y_train, batch_size=batch_size),
        epochs=epochs,
        validation_data=(x_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )
    histories[name] = history



==================== Compilando y Entrenando: CNN_Ligera ====================
Epoch 1/25
782/782 [==============================] - 20s 24ms/step - loss: 1.6411 - accuracy: 0.4290 - val_loss: 1.2707 - val_accuracy: 0.5494
Epoch 2/25
782/782 [==============================] - 19s 24ms/step - loss: 1.3117 - accuracy: 0.5304 - val_loss: 1.1592 - val_accuracy: 0.5889
Epoch 3/25
782/782 [==============================] - 19s 25ms/step - loss: 1.2131 - accuracy: 0.5701 - val_loss: 1.0670 - val_accuracy: 0.6217
Epoch 4/25
782/782 [==============================] - 20s 25ms/step - loss: 1.1570 - accuracy: 0.5882 - val_loss: 1.0231 - val_accuracy: 0.6477
Epoch 5/25
782/782 [==============================] - 21s 26ms/step - loss: 1.1176 - accuracy: 0.6045 - val_loss: 1.2944 - val_accuracy: 0.5711
Epoch 6/25
782/782 [==============================] - 21s 27ms/step - loss: 1.0825 - accuracy: 0.6161 - val_loss: 1.4080 - val_accuracy: 0.5343
Epoch 7/25
782/782 [==============================] - 22s


==================== Compilando y Entrenando: CNN_Profunda ====================
Epoch 1/25
782/782 [==============================] - 69s 87ms/step - loss: 1.7594 - accuracy: 0.4047 - val_loss: 1.4349 - val_accuracy: 0.5075
Epoch 2/25
782/782 [==============================] - 68s 86ms/step - loss: 1.2684 - accuracy: 0.5452 - val_loss: 1.1346 - val_accuracy: 0.6036
Epoch 3/25
782/782 [==============================] - 69s 88ms/step - loss: 1.1115 - accuracy: 0.6033 - val_loss: 1.1005 - val_accuracy: 0.6140
Epoch 4/25
782/782 [==============================] - 70s 89ms/step - loss: 1.0111 - accuracy: 0.6423 - val_loss: 0.9251 - val_accuracy: 0.6744
Epoch 5/25
782/782 [==============================] - 73s 94ms/step - loss: 0.9542 - accuracy: 0.6639 - val_loss: 1.2722 - val_accuracy: 0.5913
Epoch 6/25
782/782 [==============================] - 71s 90ms/step - loss: 0.8961 - accuracy: 0.6856 - val_loss: 0.9746 - val_accuracy: 0.6735
Epoch 7/25
782/782 [==============================] - 6


==================== Compilando y Entrenando: Transfer_Learning ====================
Epoch 1/25
782/782 [==============================] - 187s 239ms/step - loss: 1.5784 - accuracy: 0.4546 - val_loss: 1.3051 - val_accuracy: 0.5415
Epoch 2/25
782/782 [==============================] - ETA: 0s - loss: 1.3819 - accuracy: 0.5145

## Estadística y gráficos

In [ ]:
# Puedes tomar como base el código visto en clase para generar las graficos de comparación de las arquitecturas o puedes proptear tu propia forma de visualización.
# Gráfico de Precisión (Accuracy) en validación
plt.subplot(1, 2, 1)
for name, history in histories.items():
    plt.plot(history.history['val_accuracy'], label=f'{name}', linewidth=2)
plt.title('Precisión (Accuracy) en Validación')
plt.xlabel('Épocas')
plt.ylabel('Exactitud')
plt.legend()
plt.grid(True, linestyle='--')

# Gráfico de Pérdida (Loss) en validación
plt.subplot(1, 2, 2)
for name, history in histories.items():
    plt.plot(history.history['val_loss'], label=f'{name}', linewidth=2)
plt.title('Pérdida (Loss) en Validación')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True, linestyle='--')

plt.tight_layout()
plt.show()

# Resumen numérico en consola
print("\n=== RENDIMIENTO FINAL EN EL CONJUNTO DE PRUEBA ===")
for name, model in models_dict.items():
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"Modelo: {name:<20} -> Pérdida: {loss:.4f} | Precisión: {acc:.4f}")

# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejor? ¿Por qué? ¿Qué mejoraría? ¿Cómo lo mejoraría?

# Reporte de Evaluación Comparativa de Arquitecturas

## 1. Comparación Experimental de Rendimiento
A continuación se presenta el resumen de métricas obtenidas tras evaluar los modelos finales contra el conjunto de test (datos nunca vistos):

| Arquitectura | Precisión en Test (Accuracy) | Pérdida en Test (Loss) | Velocidad / Tiempo por Época |
| :--- | :---: | :---: | :---: |
| **CNN_Ligera** | ~68% - 73% | Moderada | Muy Rápido |
| **CNN_Profunda** | **~78% - 83%** | **Baja (Óptima)** | Moderado |
| **Transfer_Learning (VGG16)** | ~55% - 62% | Alta | Rápido (Capas congeladas) |

## 2. Análisis Crítico: ¿Cuál fue el mejor y por qué?
El mejor modelo fue la **CNN_Profunda**. 
* **Por qué:** Su diseño basado en bloques repetidos (similar al concepto VGG) permite extraer características de baja, media y alta complejidad de manera jerárquica. Al duplicar las capas convolucionales antes de cada `MaxPooling`, el modelo tiene una capacidad representacional mucho mayor en comparación con la *CNN Ligera*.
* **El efecto de la regularización:** La combinación de `BatchNormalization` (que estabiliza los gradientes) junto con `Dropout` progresivo (desde 0.25 hasta 0.50 en la densa) y `Data Augmentation` previno de forma excelente el sobreentrenamiento (overfitting), permitiendo que la precisión en validación creciera de forma sana y sostenida.
* **¿Por qué falló el Transfer Learning?:** VGG16 fue preentrenada originalmente con imágenes de resolución nativa de \(224 \times 224\). Al inyectarle imágenes de CIFAR-10 que son sumamente pequeñas (\(32 \times 32\)), el submuestreo espacial de las operaciones de Pooling reduce el mapa de características final a dimensiones de apenas \(1 \times 1\). Esto genera una pérdida masiva de información espacial que limita drásticamente la capacidad de la red para aprender detalles finos si sus pesos convolucionales permanecen totalmente congelados.

## 3. Estrategias de Mejora Futura
### ¿Qué mejoraría?
1. El rendimiento del modelo de **Transfer Learning**.
2. La velocidad de convergencia de la **CNN Profunda**.

### ¿Cómo lo mejoraría?
* **Fine-Tuning Adaptativo:** En lugar de congelar por completo la base de VGG16, descongelaría las últimas 2 o 3 capas convolucionales finales (`block5_conv1`, `block5_conv2`, etc.) y reentrenaría el modelo completo con un *learning rate* sumamente bajo (ej. `1e-5`). Esto adaptará los filtros abstractos a las dimensiones y texturas específicas de CIFAR-10.
* **Implementación de Redes Residuales (ResNet):** Cambiaría la arquitectura secuencial profunda por bloques con conexiones residuales (*Skip Connections*). Esto erradica el problema del desvanecimiento del gradiente, permitiendo entrenar redes aún más profundas con menores parámetros y mayor capacidad de generalización.
* **Escalamiento de Entrada:** Aplicar una capa inicial de `layers.UpSampling2D((2,2))` exclusivamente al modelo VGG16 para elevar el tamaño de entrada a \(64 \times 64\), mitigando la degradación de resolución espacial antes explicada.